# Round 2 PCA loadings

Diagnostic for the projection shrinkage question: plots the PCA itself, not the projected scores -- how much variance each PC actually explains (scree plot), and whether a handful of SNPs dominate a PC's loadings (Manhattan-style plot by genomic position, plus a top-N table). A PC dominated by a small genomic region (a long-range LD block that survived pruning, a structural variant, etc.) rather than genome-wide structure would explain odd projection behavior independent of anything already checked in the ID/allele-harmonization or allele-frequency diagnostics.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Inputs

In [ ]:
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/01_ancestry_filtering"
R2_OUT_DIR = f"{BUCKET_DIR}/round2_pca"

CLUSTER = "ceu_gbr"   # matches submit_pca_r2.ipynb's CLUSTER

R2_PCA_PREFIX = f"{R2_OUT_DIR}/r2_pca_{CLUSTER}"   # .eigenval / .eigenvec.allele, from submit_pca_r2.ipynb
OUT_DIR = R2_OUT_DIR

N_PCS_TO_PLOT = 6   # Manhattan plots are one per PC -- keep this modest
N_TOP_LOADINGS = 20  # how many largest-magnitude SNPs to print per PC

os.makedirs(OUT_DIR, exist_ok=True)

## Scree plot: variance explained per PC

If round 2's leading PCs explain very little variance relative to later ones, that alone would explain why projected scores along those PCs look compressed -- there's little real signal for `--score` to project onto, so noise dominates.

In [ ]:
eigenval = pd.read_csv(f"{R2_PCA_PREFIX}.eigenval", header=None, names=["eigenvalue"])
eigenval["pc"] = range(1, len(eigenval) + 1)
eigenval["pct_variance"] = eigenval["eigenvalue"] / eigenval["eigenvalue"].sum() * 100

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(eigenval["pc"], eigenval["pct_variance"], color="royalblue")
ax.set_xlabel("PC")
ax.set_ylabel("% variance explained")
ax.set_title(f"Round 2 ({CLUSTER}) scree plot")
ax.set_xticks(eigenval["pc"])
plt.tight_layout()
plot_path = os.path.join(OUT_DIR, f"scree_{CLUSTER}.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")
print(eigenval[["pc", "eigenvalue", "pct_variance"]].to_string(index=False))

## Load + parse the loadings

`eigenvec.allele` has two rows per variant (one per allele, plink2's `--pca allele-wts` convention) -- keep only the row matching the ALT allele parsed from the ID (`chr:pos:ref:alt`), so each variant contributes exactly one, sign-consistent loading per PC. `ID` column position and the `PC1`..`PCN` column range are found dynamically from the header, since plink2's exact column layout can vary by build (same approach used everywhere else in this pipeline that reads this file).

In [ ]:
weights_path = f"{R2_PCA_PREFIX}.eigenvec.allele"
weights = pd.read_csv(weights_path, sep="\t")

id_col = "ID"
a1_col = "A1"
pc_cols_raw = [f"PC{i}" for i in range(1, N_PCS_TO_PLOT + 1)]
assert id_col in weights.columns and a1_col in weights.columns, f"unexpected columns: {list(weights.columns)}"
assert all(c in weights.columns for c in pc_cols_raw), f"missing PC columns, have: {list(weights.columns)}"

id_parts = weights[id_col].str.split(":", expand=True)
weights["chr"] = id_parts[0]
weights["pos"] = id_parts[1].astype(int)
weights["alt"] = id_parts[3]

loadings = weights[weights[a1_col] == weights["alt"]].copy()
print(f"{len(weights)} total rows -> {len(loadings)} one-row-per-variant (ALT-allele) loadings")

# numeric chromosome sort order for the Manhattan plot x-axis, handling both "chr1" and "1" styles
def chr_sort_key(c):
    c = c.replace("chr", "")
    return int(c) if c.isdigit() else 100

chr_order = sorted(loadings["chr"].unique(), key=chr_sort_key)
chr_offset = {}
offset = 0
for c in chr_order:
    chr_offset[c] = offset
    offset += loadings.loc[loadings["chr"] == c, "pos"].max() + 1
loadings["plot_x"] = loadings.apply(lambda r: chr_offset[r["chr"]] + r["pos"], axis=1)

## Manhattan plot + top loadings per PC

Alternating light/dark grey by chromosome (standard Manhattan convention) so a cluster of large loadings jumps out as belonging to one region rather than being scattered genome-wide. If a PC's variance is genuinely spread across the genome, this should look like a fairly uniform scatter; a few tall spikes at one position (or one whole chromosome towering over the rest) means that PC is effectively summarizing a local LD block, not global ancestry structure -- which would explain a lot about projection behavior for that PC specifically.

In [ ]:
chr_colors = {c: ("#4a4a4a" if i % 2 == 0 else "#a0a0a0") for i, c in enumerate(chr_order)}

for pc in pc_cols_raw:
    fig, ax = plt.subplots(figsize=(12, 3))
    for c in chr_order:
        sub = loadings[loadings["chr"] == c]
        ax.scatter(sub["plot_x"], sub[pc], s=4, color=chr_colors[c], alpha=0.6, rasterized=True)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_ylabel(f"{pc} loading")
    ax.set_xlabel("Genomic position (chromosomes concatenated)")
    ax.set_title(f"Round 2 ({CLUSTER}) {pc} loadings by position")
    plt.tight_layout()
    plot_path = os.path.join(OUT_DIR, f"loadings_manhattan_{CLUSTER}_{pc}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {plot_path}")

    top = loadings.reindex(loadings[pc].abs().sort_values(ascending=False).index).head(N_TOP_LOADINGS)
    print(f"\nTop {N_TOP_LOADINGS} |{pc}| loadings:")
    print(top[["ID", "chr", "pos", pc]].to_string(index=False))
    print()